In [134]:
# Autoreload para refletir mudanças no config sem reiniciar kernel
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Célula 2 — stdlib
import pandas as pd

In [ ]:
# Célula 3 — módulo interno
from churn_telecom.config import (
    DATA_INTERIM,
    TARGET,
    get_logger,
    to_snake_case,
)

In [137]:
# Célula 4 — configurações de sessão
logger = get_logger("1.02_vab_feature_engineering")
logger.info("Iniciando feature engineering | 1.02_vab_feature_engineering")

Iniciando feature engineering | 1.02_vab_feature_engineering


In [138]:
# Célula 5 — carrega dataset limpo (saída do 1.01)
df = pd.read_parquet(DATA_INTERIM / "telco_cleaned.parquet")
logger.info("Dataset carregado | shape=%s", df.shape)
df.head(5)

Dataset carregado | shape=(7021, 20)


,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1


In [139]:
# Célula 7 — snapshot antes da engenharia de features
shape_antes = df.shape
features_antes = set(df.columns)
TARGET_COL = to_snake_case(TARGET)

logger.info(
    "SNAPSHOT ANTES | shape=%s | n_features=%d",
    shape_antes,
    shape_antes[1],
)

SNAPSHOT ANTES | shape=(7021, 20) | n_features=20


In [140]:
# Levar para o config.py depois
# Célula 8 — constantes de colunas (snake_case — alinhadas com 1.01)
# Centralizar aqui evita strings literais espalhadas pelo notebook
TENURE_COL = "tenure_months"
MONTHLY_COL = "monthly_charges"
TOTAL_COL = "total_charges"
CONTRACT_COL = "contract"
INTERNET_COL = "internet_service"
SECURITY_COL = "online_security"
SUPPORT_COL = "tech_support"

# Serviços contratados — usados para contar serviços ativos por cliente
SERVICES_COLS = [
    "phone_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
]

# Valida que todas as colunas necessárias existem no dataset
required_cols = SERVICES_COLS + [
    TENURE_COL,
    MONTHLY_COL,
    TOTAL_COL,
    CONTRACT_COL,
    INTERNET_COL,
    TARGET_COL,
]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f"Colunas ausentes para feature engineering: {missing_cols}"

logger.info("Constantes de colunas validadas | todas presentes no dataset.")

Constantes de colunas validadas | todas presentes no dataset.


In [141]:
# Célula 9 ─────────────────────────────────────────────────────────────────────
# FEATURE 1 — num_services
# ─────────────────────────────────────────────────────────────────────────────
# Contagem de serviços ativos por cliente.
# Compara com "Yes" — colunas ainda são strings originais do 1.01.
# Justificativa: clientes com mais serviços têm maior custo de saída,
# funcionando como âncora de retenção (Frontiers in AI, 2026).
# ─────────────────────────────────────────────────────────────────────────────
cols_services_present = [c for c in SERVICES_COLS if c in df.columns]

df["num_services"] = (df[cols_services_present] == "Yes").sum(axis=1).astype(int)

# log estatístico — confirma distribuição esperada
churn_por_svc = df.groupby("num_services")[TARGET_COL].mean().mul(100).round(1)

logger.info(
    "num_services | min=%d | max=%d | média=%.2f | mediana=%.1f",
    df["num_services"].min(),
    df["num_services"].max(),
    df["num_services"].mean(),
    df["num_services"].median(),
)
for n_svc, churn_rate in churn_por_svc.items():
    logger.info(
        "num_services=%d | n=%d | churn=%.1f%%",
        n_svc,
        (df["num_services"] == n_svc).sum(),
        churn_rate,
    )

num_services | min=0 | max=7 | média=2.95 | mediana=3.0
num_services=0 | n=80 | churn=43.8%
num_services=1 | n=2231 | churn=21.3%
num_services=2 | n=996 | churn=43.5%
num_services=3 | n=1041 | churn=34.7%
num_services=4 | n=1062 | churn=27.2%
num_services=5 | n=827 | churn=22.0%
num_services=6 | n=525 | churn=12.6%
num_services=7 | n=259 | churn=5.8%


In [142]:
# Célula 10 ────────────────────────────────────────────────────────────────────
# FEATURE 2 — charges_per_month
# ─────────────────────────────────────────────────────────────────────────────
# Razão entre valor mensal e tempo de casa (+1 evita divisão por zero).
# Captura clientes novos pagando muito — perfil de alto risco confirmado
# na EDA bivariada (Springer Journal of Big Data, 2024).
# ─────────────────────────────────────────────────────────────────────────────
df["charges_per_month"] = (df[MONTHLY_COL] / (df[TENURE_COL] + 1)).round(4)

logger.info(
    "charges_per_month | min=%.2f | max=%.2f | média=%.2f | mediana=%.2f",
    df["charges_per_month"].min(),
    df["charges_per_month"].max(),
    df["charges_per_month"].mean(),
    df["charges_per_month"].median(),
)

charges_per_month | min=0.26 | max=80.85 | média=5.73 | mediana=2.07


In [143]:
# Célula 11 ────────────────────────────────────────────────────────────────────
# FEATURE 3 — is_month_to_month
# ─────────────────────────────────────────────────────────────────────────────
# Flag binária: 1 se Contract == "Month-to-month".
# Maior preditor categórico do dataset (Cramer's V = 0.41, churn = 42.71%).
# Criada como int — entra no ColumnTransformer do 1.03 como passthrough.
# A coluna original "contract" permanece para o OHE do 1.03.
# ─────────────────────────────────────────────────────────────────────────────
df["is_month_to_month"] = (df[CONTRACT_COL] == "Month-to-month").astype(int)

n_mtm = df["is_month_to_month"].sum()
n_outros = len(df) - n_mtm
churn_mtm = df.loc[df["is_month_to_month"] == 1, TARGET_COL].mean() * 100

logger.info(
    "is_month_to_month | month-to-month=%d (%.1f%%) | outros=%d (%.1f%%) | "
    "churn_mtm=%.1f%%",
    n_mtm,
    n_mtm / len(df) * 100,
    n_outros,
    n_outros / len(df) * 100,
    churn_mtm,
)

is_month_to_month | month-to-month=3853 (54.9%) | outros=3168 (45.1%) | churn_mtm=42.6%


In [144]:
# Célula 12 ────────────────────────────────────────────────────────────────────
# FEATURE 4 — tenure_group
# ─────────────────────────────────────────────────────────────────────────────
# Segmentação do tempo de casa em 3 buckets:
#   novo  : ≤ 12 meses  — zona crítica (mediana churn = 10m)
#   medio : 13–48 meses — estabilização do relacionamento
#   longo : > 48 meses  — clientes fidelizados
# Captura a não-linearidade identificada na EDA bivariada:
# Cohen's d = -0.89, concentrado nos primeiros 12 meses (Nature, 2025).
# Tipada como string — OHE aplicado no ColumnTransformer do 1.03.
# ─────────────────────────────────────────────────────────────────────────────
df["tenure_group"] = pd.cut(
    df[TENURE_COL],
    bins=[0, 12, 48, float("inf")],
    labels=["novo", "medio", "longo"],
    right=True,
    include_lowest=True,
).astype(str)

logger.info("tenure_group | distribuição e churn por bucket:")
for grupo in ["novo", "medio", "longo"]:
    mask = df["tenure_group"] == grupo
    count = mask.sum()
    churn_rate = df.loc[mask, TARGET_COL].mean() * 100
    logger.info(
        "tenure_group='%s' | n=%d (%.1f%%) | churn=%.1f%%",
        grupo,
        count,
        count / len(df) * 100,
        churn_rate,
    )

tenure_group | distribuição e churn por bucket:
tenure_group='novo' | n=2164 (30.8%) | churn=47.4%
tenure_group='medio' | n=2618 (37.3%) | churn=23.6%
tenure_group='longo' | n=2239 (31.9%) | churn=9.5%


In [145]:
# Célula 13 ────────────────────────────────────────────────────────────────────
# FEATURE 5 — has_security_support
# ─────────────────────────────────────────────────────────────────────────────
# Flag binária: 1 se tem online_security OU tech_support.
# Ambas com Cramer's V ~0.34 e churn > 41% quando ausentes.
# Compara com "Yes" — colunas ainda são strings do 1.01.
# Consolidar duas features correlacionadas em uma reduz dimensionalidade
# sem perda de sinal (Frontiers in AI, 2026).
# ─────────────────────────────────────────────────────────────────────────────
df["has_security_support"] = (
    (df[SECURITY_COL] == "Yes") | (df[SUPPORT_COL] == "Yes")
).astype(int)

n_has = df["has_security_support"].sum()
n_nao_tem = len(df) - n_has
churn_sem = df.loc[df["has_security_support"] == 0, TARGET_COL].mean() * 100
churn_com = df.loc[df["has_security_support"] == 1, TARGET_COL].mean() * 100

logger.info(
    "has_security_support | tem=%d (%.1f%%) churn=%.1f%% | "
    "não tem=%d (%.1f%%) churn=%.1f%%",
    n_has,
    n_has / len(df) * 100,
    churn_com,
    n_nao_tem,
    n_nao_tem / len(df) * 100,
    churn_sem,
)

has_security_support | tem=2964 (42.2%) churn=17.1% | não tem=4057 (57.8%) churn=33.3%


In [146]:
# Célula 14 ────────────────────────────────────────────────────────────────────
# FEATURE 6 — is_fiber_optic
# ─────────────────────────────────────────────────────────────────────────────
# Flag binária: 1 se Internet Service == "Fiber optic".
# Maior taxa de churn por categoria de internet (41.89%, Cramer's V = 0.32).
# Isola o sinal mais forte do internet_service em uma feature binária.
# ─────────────────────────────────────────────────────────────────────────────
df["is_fiber_optic"] = (df[INTERNET_COL] == "Fiber optic").astype(int)
print(df["is_fiber_optic"].value_counts())
n_fiber = df["is_fiber_optic"].sum()
n_outros = len(df) - n_fiber
churn_fiber = df.loc[df["is_fiber_optic"] == 1, TARGET_COL].mean() * 100
churn_rest = df.loc[df["is_fiber_optic"] == 0, TARGET_COL].mean() * 100

logger.info(
    "is_fiber_optic | fiber=%d (%.1f%%) churn=%.1f%% | outros=%d (%.1f%%) churn=%.1f%%",
    n_fiber,
    n_fiber / len(df) * 100,
    churn_fiber,
    n_outros,
    n_outros / len(df) * 100,
    churn_rest,
)

is_fiber_optic | fiber=3090 (44.0%) churn=41.8% | outros=3931 (56.0%) churn=14.4%


is_fiber_optic
0    3931
1    3090
Name: count, dtype: int64


In [147]:
# Célula 16 — validações das features criadas
NEW_FEATURES = [
    "num_services",
    "charges_per_month",
    "is_month_to_month",
    "tenure_group",
    "has_security_support",
    "is_fiber_optic",
]

# todas as features foram criadas
for feat in NEW_FEATURES:
    assert feat in df.columns, f"Feature ausente após criação: {feat}"

# sem nulos nas features novas
nulos_novos = df[NEW_FEATURES].isnull().sum()
nulos_com_problema = nulos_novos[nulos_novos > 0]
assert len(nulos_com_problema) == 0, (
    f"Nulos em features novas: {nulos_com_problema.to_dict()}"
)

# features binárias contêm apenas 0 e 1
BINARY_NEW_FEATURES = [
    "is_month_to_month",
    "has_security_support",
    "is_fiber_optic",
]
for feat in BINARY_NEW_FEATURES:
    valores_unicos = set(df[feat].unique())
    assert valores_unicos.issubset({0, 1}), (
        f"{feat} contém valores inesperados: {valores_unicos}"
    )

# num_services dentro do range esperado
assert df["num_services"].between(0, len(SERVICES_COLS)).all(), (
    f"num_services fora do range esperado [0, {len(SERVICES_COLS)}]"
)

# colunas binárias originais ainda são strings — encoding NÃO foi feito aqui
for col in ["online_security", "tech_support", "partner"]:
    assert df[col].dtype in ["object", "category"], (
        f"Coluna '{col}' foi encodada indevidamente no 1.02 — "
        "encoding é responsabilidade do 1.03."
    )

logger.info(
    "Validações OK | %d features criadas | strings originais preservadas.",
    len(NEW_FEATURES),
)

Validações OK | 6 features criadas | strings originais preservadas.


In [148]:
# Célula 17 — correlação das features novas com o target
NUMERIC_NEW = [
    "num_services",
    "charges_per_month",
    "is_month_to_month",
    "has_security_support",
    "is_fiber_optic",
]

logger.info("=== CORRELAÇÃO FEATURES NOVAS vs TARGET ===")
correlacoes = {feat: round(df[feat].corr(df[TARGET_COL]), 4) for feat in NUMERIC_NEW}
for feat, corr in sorted(correlacoes.items(), key=lambda x: abs(x[1]), reverse=True):
    logger.info("%s | corr_target=%.4f", feat, corr)

=== CORRELAÇÃO FEATURES NOVAS vs TARGET ===
charges_per_month | corr_target=0.4094
is_month_to_month | corr_target=0.4049
is_fiber_optic | corr_target=0.3082
has_security_support | corr_target=-0.1817
num_services | corr_target=-0.0842


In [149]:
# Célula 18 — resumo final e persistência
features_depois = set(df.columns)
features_novas = features_depois - features_antes

logger.info(
    "RESUMO | shape: %s → %s | features novas=%d | %s",
    shape_antes,
    df.shape,
    len(features_novas),
    sorted(features_novas),
)

# persistência
output_path = DATA_INTERIM / "telco_features.parquet"
df.to_parquet(output_path, index=False)

# validação pós-save
df_check = pd.read_parquet(output_path)
assert df_check.shape == df.shape, (
    f"Erro na persistência | esperado={df.shape} | lido={df_check.shape}"
)

logger.info(
    "Dataset salvo e validado | path=%s | shape=%s",
    output_path,
    df.shape,
)
logger.info("1.02 feature_engineering concluído com sucesso.")

RESUMO | shape: (7021, 20) → (7021, 26) | features novas=6 | ['charges_per_month', 'has_security_support', 'is_fiber_optic', 'is_month_to_month', 'num_services', 'tenure_group']
Dataset salvo e validado | path=C:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\projeto_final\data\interim\telco_features.parquet | shape=(7021, 26)
1.02 feature_engineering concluído com sucesso.
